# Figure S6 — integrated annotation of cNMF programs across D0–D3

This driver reproduces Figure S6 by executing the **existing all-day program-annotation notebook**:

`program_annotation_panels_D0_D1_D2_D3_rotated_logos_D2_extra_D3_trimmed_markers.ipynb`

The source notebook's selection rules, heatmap geometry, shared scales, significance stars, rotated vector motif logos, and export code are retained. The driver changes only input/output path assignments. In particular, every motif-enrichment value is forced to come from:

`/oak/stanford/groups/engreitz/Users/opushkar/multiome_ipsc_ec/analysis/2.tf_enrichment_in_programs/enhancer_enrichment_results.tsv`

The older `multiome_ipsc_ec_v0` and `enhancer_enrichment_results_min10.tsv` tables are rejected.


In [ ]:
from pathlib import Path
import json
import os
import sys

def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from within the "
        "cc-perturbseq-manuscript-analysis checkout."
    )

REPO_ROOT = _find_repo_root()
for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from src.cc_perturbseq.figure_notebook import (
    initialize_figure,
    panel_directory,
    write_receipt,
)
from src.cc_perturbseq.motif_sources import (
    canonical_enhancer_enrichment_path,
    canonical_motif_logo_inputs,
)
from src.cc_perturbseq.source_notebook_runner import (
    execute_notebook,
    patch_source_notebook,
    sha256_file,
)

FIGURE = {
    "figure_id": "Figure_S06",
    "title": "Figure S6 — Integrated annotation of cNMF programs across D0–D3",
    "scope": "All day-grouped program-annotation panels, source-notebook execution",
    "manuscript": "Working CC-Perturb-seq EC Differentiation - Text V3",
    "panels": [{
        "panel": "all",
        "description": (
            "Aligned regulator-effect, cNMF-loading and motif-enrichment "
            "panels for D0–D3"
        ),
        "type": "source_notebook",
        "required_inputs": [],
        "candidate_sources": [
            (
                "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/"
                "notebooks/program_annotation_panels_D0_D1_D2_D3_"
                "rotated_logos_D2_extra_D3_trimmed_markers.ipynb"
            )
        ],
    }],
}
ctx = initialize_figure(FIGURE)
OUTDIR = panel_directory(ctx, "all")


In [ ]:
EXPECTED_SOURCE_SHA256 = (
    "b0ea563bee36408e54ff793e4915a7258232473a65813bba2bb5d7b5f655599a"
)
SOURCE_ENV = "CCPERTURBSEQ_FIGURE_S06_SOURCE_NOTEBOOK"
oak_source = Path(
    "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/"
    "program_annotation_panels_D0_D1_D2_D3_rotated_logos_D2_extra_"
    "D3_trimmed_markers.ipynb"
)
archive_source = (
    REPO_ROOT
    / "recovery"
    / "sherlock_notebooks"
    / "oak"
    / "stanford"
    / "groups"
    / "engreitz"
    / "Users"
    / "tri"
    / "CodeBase"
    / "programDE"
    / "notebooks"
    / "program_annotation_panels_D0_D1_D2_D3_rotated_logos_D2_extra_"
      "D3_trimmed_markers.ipynb"
)

def _usable_notebook(path):
    if not path or not path.is_file() or path.stat().st_size < 1024:
        return False
    prefix = path.read_bytes()[:200]
    return b"git-lfs.github.com/spec" not in prefix

explicit = os.environ.get(SOURCE_ENV)
candidates = ([Path(explicit).expanduser()] if explicit else []) + [
    oak_source,
    archive_source,
]
SOURCE_NOTEBOOK = next(
    (path.resolve() for path in candidates if _usable_notebook(path)),
    None,
)
if SOURCE_NOTEBOOK is None:
    raise FileNotFoundError(
        "The exact Figure S6 source notebook was not found as materialized "
        "bytes. Checked: " + ", ".join(str(path) for path in candidates)
        + f". Set {SOURCE_ENV} to the exact source notebook."
    )

observed_sha = sha256_file(SOURCE_NOTEBOOK)
if observed_sha != EXPECTED_SOURCE_SHA256:
    raise ValueError(
        "Figure S6 source notebook drifted from the recovered version. "
        f"Expected {EXPECTED_SOURCE_SHA256}; observed {observed_sha}; "
        f"path {SOURCE_NOTEBOOK}."
    )

ENRICHMENT_PATH = canonical_enhancer_enrichment_path()
CLUSTER_MAP_PATH, MODISCO_H5_PATH = canonical_motif_logo_inputs()
display({
    "source_notebook": str(SOURCE_NOTEBOOK),
    "source_sha256": observed_sha,
    "enhancer_enrichment": str(ENRICHMENT_PATH),
    "cluster_map": str(CLUSTER_MAP_PATH),
    "modisco_h5": str(MODISCO_H5_PATH),
    "output_directory": str(OUTDIR),
})


In [ ]:
PATCHED_NOTEBOOK = OUTDIR / "Figure_S06_source_patched.ipynb"
EXECUTED_NOTEBOOK = OUTDIR / "Figure_S06_source_executed.ipynb"

prepared, provenance = patch_source_notebook(
    SOURCE_NOTEBOOK,
    PATCHED_NOTEBOOK,
    expected_sha256=EXPECTED_SOURCE_SHA256,
    path_replacements={
        "ENHANCER_ENRICHMENT_PATH": ENRICHMENT_PATH,
        "CLUSTER_MAP_PATH": CLUSTER_MAP_PATH,
        "MODISCO_H5_PATH": MODISCO_H5_PATH,
        "OUTDIR": OUTDIR,
    },
)
provenance_path = OUTDIR / "Figure_S06_source_provenance.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(provenance)


In [ ]:
execute_notebook(
    prepared,
    EXECUTED_NOTEBOOK,
    working_directory=SOURCE_NOTEBOOK.parent,
    timeout=14_400,
)

generated = sorted(
    path for path in OUTDIR.iterdir()
    if path.is_file() and path not in {PATCHED_NOTEBOOK, EXECUTED_NOTEBOOK}
)
write_receipt(
    ctx,
    "all",
    status="EXACT_SOURCE_NOTEBOOK_EXECUTED_WITH_PATH_ONLY_SUBSTITUTIONS",
    inputs={
        "source_notebook": str(SOURCE_NOTEBOOK),
        "source_sha256": EXPECTED_SOURCE_SHA256,
        "enhancer_enrichment_results": str(ENRICHMENT_PATH),
        "cluster_motif_map": str(CLUSTER_MAP_PATH),
        "modisco_compiled_h5": str(MODISCO_H5_PATH),
    },
    outputs={
        "patched_source_notebook": str(PATCHED_NOTEBOOK),
        "executed_source_notebook": str(EXECUTED_NOTEBOOK),
        "provenance": str(provenance_path),
        **{
            f"generated_{index:03d}": str(path)
            for index, path in enumerate(generated, start=1)
        },
    },
    parameters={
        "source_logic_changed": False,
        "path_assignments_changed": [
            "ENHANCER_ENRICHMENT_PATH",
            "CLUSTER_MAP_PATH",
            "MODISCO_H5_PATH",
            "OUTDIR",
        ],
        "motif_enrichment_path_locked": True,
    },
)
print("Figure S6 outputs:", OUTDIR)
for path in generated:
    print(path.name)
